# Compilation Strategies Deep Dive

qb-compiler provides five built-in compilation strategies, each optimising
for a different objective.  A *strategy* decides which compiler passes to
run and in what order.

| Strategy | Optimises for | Key trade-off |
|----------|-------------|---------------|
| `fidelity_optimal` | Output fidelity | Slower compile, uses calibration |
| `depth_optimal` | Minimum circuit depth | May not use best qubits |
| `speed_optimal` | Compilation speed | No calibration, basic passes only |
| `cost_optimal` | Fidelity per dollar | Prefers cheaper qubits when fidelity is similar |
| `budget_aware` | Fidelity within budget | Reduces shots or raises error if over budget |

All strategies implement the `CompilationStrategy` abstract interface and
produce a `PassManager` --- an ordered list of compiler passes.

### Two API levels

- **High-level**: `QBCompiler.compile(circuit, strategy=...)` supports three
  strategies: `fidelity_optimal`, `depth_optimal`, and `budget_optimal`.
- **Low-level**: `get_strategy(name)` → `strategy.build_pass_manager(config)`
  gives access to all five strategies, including `speed_optimal` and
  `cost_optimal`.

In [1]:
from qb_compiler import QBCircuit, QBCompiler, CompilerConfig
from qb_compiler.strategies import (
    get_strategy,
    CompilationStrategy,
    PassManager,
    FidelityOptimalStrategy,
    DepthOptimalStrategy,
    SpeedOptimalStrategy,
    CostOptimalStrategy,
    BudgetAwareStrategy,
)

## 1. The `CompilationStrategy` interface

Every strategy inherits from `CompilationStrategy` and must implement:

- `build_pass_manager(config, calibration, noise_model) -> PassManager`
- `name -> str` (property)

The `PassManager` is a lightweight container of `PassConfig` objects,
each describing a single compiler pass with its options.

In [2]:
# The strategy interface
import inspect

print("CompilationStrategy abstract methods:")
for name, method in inspect.getmembers(CompilationStrategy, predicate=inspect.isfunction):
    if getattr(method, '__isabstractmethod__', False):
        print(f"  {name}{inspect.signature(method)}")

# Abstract properties
for name in dir(CompilationStrategy):
    attr = getattr(CompilationStrategy, name, None)
    if isinstance(attr, property) and getattr(attr.fget, '__isabstractmethod__', False):
        print(f"  @property {name}")

CompilationStrategy abstract methods:
  build_pass_manager(self, config: 'CompilerConfig', calibration: 'CalibrationProvider | None' = None, noise_model: 'NoiseModel | None' = None) -> 'PassManager'
  @property name


## 2. Looking up strategies by name

The `get_strategy()` factory creates a strategy instance by name.
Short aliases (`"fidelity"`, `"speed"`, etc.) are accepted.

In [3]:
# Look up by name
for name in ["fidelity_optimal", "depth_optimal", "speed_optimal", "cost_optimal"]:
    s = get_strategy(name)
    print(f"{name:>20} -> {type(s).__name__} (name='{s.name}')")

# Short aliases work too
assert get_strategy("fidelity").name == "fidelity_optimal"
assert get_strategy("speed").name == "speed_optimal"
print("\nShort aliases verified.")

    fidelity_optimal -> FidelityOptimalStrategy (name='fidelity_optimal')
       depth_optimal -> DepthOptimalStrategy (name='depth_optimal')
       speed_optimal -> SpeedOptimalStrategy (name='speed_optimal')
        cost_optimal -> CostOptimalStrategy (name='cost_optimal')

Short aliases verified.


## 3. FidelityOptimalStrategy

The default and most comprehensive strategy.  Uses calibration-aware
layout, noise-aware routing, gate cancellation, commutation analysis,
and T1/T2-aware scheduling.

### Pass pipeline

In [4]:
config = CompilerConfig(backend="ibm_fez", optimization_level=2, seed=42)
strategy = FidelityOptimalStrategy()

pm = strategy.build_pass_manager(config)
print(pm.describe())

PassManager (8 passes):
  1. [analysis] circuit_analysis {'collect_gate_counts': True, 'collect_depth': True}
  2. [optimization] gate_cancellation {'max_iterations': 2}
  3. [optimization] commutation_analysis {'enabled': True}
  4. [decomposition] basis_translation {'target_basis': ['id', 'rz', 'sx', 'x', 'cz', 'reset']}
  5. [routing] trivial_layout
  6. [optimization] post_route_cancellation {'max_iterations': 3}
  7. [scheduling] alap_scheduling {'strategy': 'alap'}
  8. [analysis] fidelity_estimation {'estimate_fidelity': True}


### Effect of optimization level

Higher optimization levels enable more aggressive passes (commutation
analysis, peephole optimization) at the cost of compile time.

In [5]:
for opt_level in [0, 1, 2, 3]:
    cfg = CompilerConfig(backend="ibm_fez", optimization_level=opt_level)
    pm = strategy.build_pass_manager(cfg)
    print(f"opt_level={opt_level}: {len(pm)} passes")
    for p in pm:
        print(f"    [{p.pass_type}] {p.name}")
    print()

opt_level=0: 5 passes
    [analysis] circuit_analysis
    [decomposition] basis_translation
    [routing] trivial_layout
    [scheduling] alap_scheduling
    [analysis] fidelity_estimation

opt_level=1: 7 passes
    [analysis] circuit_analysis
    [optimization] gate_cancellation
    [optimization] commutation_analysis
    [decomposition] basis_translation
    [routing] trivial_layout
    [scheduling] alap_scheduling
    [analysis] fidelity_estimation

opt_level=2: 8 passes
    [analysis] circuit_analysis
    [optimization] gate_cancellation
    [optimization] commutation_analysis
    [decomposition] basis_translation
    [routing] trivial_layout
    [optimization] post_route_cancellation
    [scheduling] alap_scheduling
    [analysis] fidelity_estimation

opt_level=3: 9 passes
    [analysis] circuit_analysis
    [optimization] gate_cancellation
    [optimization] commutation_analysis
    [decomposition] basis_translation
    [routing] trivial_layout
    [optimization] post_route_cance

## 4. DepthOptimalStrategy

Aggressively minimises circuit depth using repeated gate cancellation,
commutation-based reordering, circuit simplification, and ASAP scheduling.
Useful when decoherence (T2) is the dominant error source.

In [6]:
depth_strategy = DepthOptimalStrategy()
pm_depth = depth_strategy.build_pass_manager(config)
print(pm_depth.describe())

PassManager (10 passes):
  1. [analysis] circuit_analysis {'collect_gate_counts': True, 'collect_depth': True}
  2. [optimization] gate_cancellation {'max_iterations': 8}
  3. [optimization] commutation_analysis {'enabled': True, 'objective': 'minimize_depth'}
  4. [optimization] circuit_simplification {'merge_single_qubit_gates': True, 'remove_identity': True}
  5. [decomposition] basis_translation {'target_basis': ['id', 'rz', 'sx', 'x', 'cz', 'reset']}
  6. [routing] trivial_layout
  7. [optimization] post_route_cancellation {'max_iterations': 5}
  8. [optimization] commutation_analysis {'enabled': True, 'objective': 'minimize_depth'}
  9. [scheduling] asap_scheduling {'strategy': 'asap'}
  10. [analysis] depth_analysis {'collect_depth': True}


Note the differences from fidelity_optimal:
- Gate cancellation with `max_iterations=8` (vs 2-5 for fidelity)
- Commutation analysis with `objective="minimize_depth"`
- Circuit simplification pass (merge single-qubit gates, remove identity)
- ASAP scheduling (vs ALAP for fidelity)
- No noise-aware layout or routing

## 5. SpeedOptimalStrategy

The minimal pipeline: trivial layout, basic routing, basis decomposition.
No calibration, no optimisation passes, no noise modelling.  Use this for
development and simulator runs.

In [7]:
speed_strategy = SpeedOptimalStrategy()
pm_speed = speed_strategy.build_pass_manager(config)
print(pm_speed.describe())
print(f"\nOnly {len(pm_speed)} passes --- minimal overhead.")

PassManager (2 passes):
  1. [routing] trivial_layout
  2. [decomposition] basis_translation {'target_basis': ['id', 'rz', 'sx', 'x', 'cz', 'reset']}

Only 2 passes --- minimal overhead.


## 6. CostOptimalStrategy

Similar to fidelity_optimal, but factors in execution cost.  When two
layout options have comparable fidelity, it prefers the cheaper one.
Optimises for fidelity-per-dollar.

In [8]:
cost_strategy = CostOptimalStrategy()
pm_cost = cost_strategy.build_pass_manager(config)
print(pm_cost.describe())

PassManager (9 passes):
  1. [analysis] circuit_analysis {'collect_gate_counts': True, 'collect_depth': True}
  2. [analysis] cost_analysis {'cost_per_shot': 0.00016, 'backend': 'ibm_fez'}
  3. [optimization] gate_cancellation {'max_iterations': 3}
  4. [optimization] commutation_analysis {'enabled': True}
  5. [decomposition] basis_translation {'target_basis': ['id', 'rz', 'sx', 'x', 'cz', 'reset']}
  6. [routing] trivial_layout
  7. [optimization] post_route_cancellation {'max_iterations': 3}
  8. [scheduling] alap_scheduling {'strategy': 'alap'}
  9. [analysis] fidelity_estimation {'estimate_fidelity': True, 'include_cost': True}


Note the `cost_analysis` pass and `cost_aware_layout` --- these are
unique to the cost strategy.

## 7. BudgetAwareStrategy

Takes an explicit USD budget and requested shot count.  If the estimated
cost exceeds the budget, it reduces shots to fit.  If even a single shot
is too expensive, it raises `BudgetExceededError`.

In [9]:
# Budget of $5 with 10,000 shots on IBM Fez
budget_strategy = BudgetAwareStrategy(budget_usd=5.0, shots=10_000)
pm_budget = budget_strategy.build_pass_manager(config)
print(pm_budget.describe())

print(f"\nRequested shots:  10,000")
print(f"Effective shots:  {budget_strategy.effective_shots:,}")
print(f"Budget:           ${budget_strategy.budget_usd:.2f}")

PassManager (7 passes):
  1. [analysis] circuit_analysis {'collect_gate_counts': True, 'collect_depth': True}
  2. [analysis] budget_constraint {'budget_usd': 5.0, 'effective_shots': 10000, 'backend': 'ibm_fez'}
  3. [optimization] gate_cancellation {'max_iterations': 2}
  4. [decomposition] basis_translation {'target_basis': ['id', 'rz', 'sx', 'x', 'cz', 'reset']}
  5. [routing] trivial_layout
  6. [optimization] post_route_cancellation {'max_iterations': 3}
  7. [scheduling] alap_scheduling {'strategy': 'alap'}

Requested shots:  10,000
Effective shots:  10,000
Budget:           $5.00


In [10]:
# What happens with a tight budget on an expensive backend?
from qb_compiler.exceptions import BudgetExceededError

ionq_config = CompilerConfig(backend="ionq_aria", optimization_level=2)

# IonQ Aria: $0.30/shot.  100 shots = $30.  Budget = $10.
tight_budget = BudgetAwareStrategy(budget_usd=10.0, shots=100)
pm_tight = tight_budget.build_pass_manager(ionq_config)
print(f"IonQ Aria, $10 budget, requested 100 shots")
print(f"  Effective shots: {tight_budget.effective_shots}")
print(f"  (reduced from 100 to fit within budget)")

# Budget so tight that even 1 shot is too expensive
try:
    impossible = BudgetAwareStrategy(budget_usd=0.10, shots=100)
    impossible.build_pass_manager(ionq_config)
except BudgetExceededError as e:
    print(f"\nBudgetExceededError: {e}")

IonQ Aria, $10 budget, requested 100 shots
  Effective shots: 100
  (reduced from 100 to fit within budget)


## 8. Strategy comparison on the same circuit

Let's compile the same circuit with the three strategies supported by the
high-level `QBCompiler.compile()` API (`fidelity_optimal`, `depth_optimal`,
`budget_optimal`), then show how `speed_optimal` and `cost_optimal` are
accessed via the lower-level strategy interface.

In [11]:
import time

# Build a 6-qubit circuit with mixed gate types
circ = QBCircuit(6)
circ.h(0).h(1)
for i in range(5):
    circ.cx(i, i + 1)
for i in range(6):
    circ.rz(i, 0.3)
for i in range(4, -1, -1):
    circ.cx(i, i + 1)
circ.measure_all()

print(f"Input circuit: {circ}")
print(f"  gate_count={circ.gate_count}, depth={circ.depth}, 2Q={circ.two_qubit_count}")

# --- High-level API: the 3 strategies supported by QBCompiler.compile() ---
compiler = QBCompiler.from_backend("ibm_fez")

print(f"\n{'Strategy':>20} {'Depth':>6} {'Gates':>6} {'Fidelity':>10} {'Time (ms)':>10}")
print("-" * 60)

for strategy_name in ["fidelity_optimal", "depth_optimal", "budget_optimal"]:
    t0 = time.perf_counter()
    result = compiler.compile(circ, strategy=strategy_name)
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"{strategy_name:>20} {result.compiled_depth:>6} "
          f"{result.compiled_circuit.gate_count:>6} "
          f"{result.estimated_fidelity:>10.4f} {elapsed:>10.1f}")

# --- Low-level API: speed_optimal and cost_optimal via get_strategy() ---
# These strategies are not wired into QBCompiler.compile(), so we use them
# directly through the strategy → PassManager interface.
config = CompilerConfig(backend="ibm_fez", optimization_level=2, seed=42)

for strategy_name in ["speed_optimal", "cost_optimal"]:
    strat = get_strategy(strategy_name)
    t0 = time.perf_counter()
    pm = strat.build_pass_manager(config)
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"{strategy_name:>20} {'-':>6} {'-':>6} {'-':>10} {elapsed:>10.1f}  [pass manager only]")

Input circuit: QBCircuit(n_qubits=6, ops=24, depth=13)
  gate_count=24, depth=13, 2Q=10

            Strategy  Depth  Gates   Fidelity  Time (ms)
------------------------------------------------------------


    fidelity_optimal     48     86     0.9195     4808.7


       depth_optimal     48     86     0.9195     1618.0
      budget_optimal     51     88     0.8638        0.7
       speed_optimal      -      -          -        0.0  [pass manager only]
        cost_optimal      -      -          -        0.0  [pass manager only]


## 9. When to use which strategy

| Use case | Recommended strategy |
|----------|---------------------|
| Production runs on real hardware | `fidelity_optimal` |
| Short-T2 backends (Rigetti, IQM) | `depth_optimal` |
| Development / simulator testing | `speed_optimal` |
| Multi-backend cost comparison | `cost_optimal` |
| Constrained research budgets | `budget_aware` |
| Interactive exploration | `speed_optimal` (fast iteration) |

## 10. Custom strategy composition

You can create a custom strategy by subclassing `CompilationStrategy`
and building your own pass pipeline.

In [12]:
from qb_compiler.strategies.base import PassConfig

class CustomHybridStrategy(CompilationStrategy):
    """Custom strategy: aggressive depth reduction + noise-aware scheduling."""

    @property
    def name(self) -> str:
        return "custom_hybrid"

    def build_pass_manager(self, config, calibration=None, noise_model=None):
        pm = PassManager()

        # Start with aggressive gate cancellation (from depth_optimal)
        pm.append(PassConfig(
            name="gate_cancellation",
            pass_type="optimization",
            options={"max_iterations": 8},
        ))

        # Add circuit simplification
        pm.append(PassConfig(
            name="circuit_simplification",
            pass_type="optimization",
            options={"merge_single_qubit_gates": True},
        ))

        # Basis decomposition
        basis = config.effective_basis_gates
        if basis:
            pm.append(PassConfig(
                name="basis_translation",
                pass_type="decomposition",
                options={"target_basis": list(basis)},
            ))

        # Layout and routing
        pm.append(PassConfig(
            name="trivial_layout",
            pass_type="routing",
        ))
        if config.coupling_map:
            pm.append(PassConfig(
                name="swap_routing",
                pass_type="routing",
                options={"coupling_map": config.coupling_map, "method": "sabre"},
            ))

        # Noise-aware scheduling (from fidelity_optimal)
        pm.append(PassConfig(
            name="t2_aware_scheduling",
            pass_type="scheduling",
            options={"strategy": "alap_noise_aware"},
        ))

        return pm


custom = CustomHybridStrategy()
pm_custom = custom.build_pass_manager(config)
print(f"Custom strategy: '{custom.name}'")
print(pm_custom.describe())

Custom strategy: 'custom_hybrid'
PassManager (5 passes):
  1. [optimization] gate_cancellation {'max_iterations': 8}
  2. [optimization] circuit_simplification {'merge_single_qubit_gates': True}
  3. [decomposition] basis_translation {'target_basis': ['id', 'rz', 'sx', 'x', 'cz', 'reset']}
  4. [routing] trivial_layout
  5. [scheduling] t2_aware_scheduling {'strategy': 'alap_noise_aware'}


## 11. Pass pipeline inspection

Each strategy's pass manager can be inspected programmatically.  This is
useful for understanding what the compiler will do before running it.

In [13]:
from collections import Counter

strategies = {
    "fidelity": FidelityOptimalStrategy(),
    "depth":    DepthOptimalStrategy(),
    "speed":    SpeedOptimalStrategy(),
    "cost":     CostOptimalStrategy(),
    "budget":   BudgetAwareStrategy(budget_usd=100.0, shots=4096),
}

print(f"{'Strategy':>12} {'Total':>6} {'analysis':>9} {'optim':>6} {'routing':>8} {'sched':>6} {'decomp':>7}")
print("-" * 62)

for name, strat in strategies.items():
    pm = strat.build_pass_manager(config)
    counts = Counter(p.pass_type for p in pm)
    print(f"{name:>12} {len(pm):>6} "
          f"{counts.get('analysis', 0):>9} "
          f"{counts.get('optimization', 0):>6} "
          f"{counts.get('routing', 0):>8} "
          f"{counts.get('scheduling', 0):>6} "
          f"{counts.get('decomposition', 0):>7}")

    Strategy  Total  analysis  optim  routing  sched  decomp
--------------------------------------------------------------
    fidelity      8         2      3        1      1       1
       depth     10         2      5        1      1       1
       speed      2         0      0        1      0       1
        cost      9         3      3        1      1       1
      budget      7         2      2        1      1       1


## 12. Trade-offs: fidelity vs depth vs cost vs compile time

The strategies represent different points on a multi-objective trade-off
surface.  Here we compare the three `compile()`-supported strategies
across multiple circuit families.

In [14]:
import time

# Build three different circuit types
circuits = {}

# GHZ: linear chain of CX
circuits["GHZ-8"] = QBCircuit(8).h(0)
for i in range(7):
    circuits["GHZ-8"].cx(i, i + 1)
circuits["GHZ-8"].measure_all()

# QAOA layer
qaoa = QBCircuit(8)
for i in range(7):
    qaoa.cx(i, i + 1).rz(i + 1, 0.5).cx(i, i + 1)
for i in range(8):
    qaoa.rx(i, 0.7)
qaoa.measure_all()
circuits["QAOA-8"] = qaoa

# Random-looking circuit with varied gates
mixed = QBCircuit(8)
for i in range(8):
    mixed.h(i)
for i in range(0, 8, 2):
    if i + 1 < 8:
        mixed.cx(i, i + 1)
for i in range(8):
    mixed.rz(i, 0.3 * i)
for i in range(1, 8, 2):
    if i + 1 < 8:
        mixed.cx(i, i + 1)
mixed.measure_all()
circuits["Mixed-8"] = mixed

compiler = QBCompiler.from_backend("ibm_fez")

for circ_name, circ in circuits.items():
    print(f"\n=== {circ_name} (gates={circ.gate_count}, depth={circ.depth}) ===")
    print(f"{'Strategy':>18} {'Depth':>6} {'Fidelity':>10} {'Time (ms)':>10}")
    print("-" * 50)

    for strat_name in ["fidelity_optimal", "depth_optimal", "budget_optimal"]:
        t0 = time.perf_counter()
        result = compiler.compile(circ, strategy=strat_name)
        elapsed = (time.perf_counter() - t0) * 1000
        print(f"{strat_name:>18} {result.compiled_depth:>6} "
              f"{result.estimated_fidelity:>10.4f} {elapsed:>10.1f}")


=== GHZ-8 (gates=16, depth=9) ===
          Strategy  Depth   Fidelity  Time (ms)
--------------------------------------------------


  fidelity_optimal     31     0.9042     2585.1


     depth_optimal     31     0.9042     2455.7
    budget_optimal     32     0.8711        0.5

=== QAOA-8 (gates=37, depth=23) ===
          Strategy  Depth   Fidelity  Time (ms)
--------------------------------------------------


  fidelity_optimal     74     0.8831     2368.0


     depth_optimal     74     0.8831     2240.6
    budget_optimal     89     0.8187        0.7

=== Mixed-8 (gates=31, depth=5) ===
          Strategy  Depth   Fidelity  Time (ms)
--------------------------------------------------


  fidelity_optimal     15     0.8975     2258.4


     depth_optimal     15     0.8975     1632.5
    budget_optimal     16     0.8586        0.4


## 13. Benchmarking strategies across circuit families

Systematic comparison of the three `compile()`-supported strategies
across increasing circuit sizes.

In [15]:
import time

sizes = [4, 6, 8, 10, 12]

print(f"{'Size':>6} {'Fidelity depth':>16} {'Depth depth':>14} {'Budget depth':>14}")
print("-" * 55)

for n in sizes:
    # Build a chain circuit
    circ = QBCircuit(n)
    for i in range(n - 1):
        circ.cx(i, i + 1).rz(i + 1, 0.5).cx(i, i + 1)
    for i in range(n):
        circ.rx(i, 0.7)
    circ.measure_all()

    r_fid = compiler.compile(circ, strategy="fidelity_optimal")
    r_dep = compiler.compile(circ, strategy="depth_optimal")
    r_bud = compiler.compile(circ, strategy="budget_optimal")

    print(f"{n:>6} {r_fid.compiled_depth:>16} {r_dep.compiled_depth:>14} {r_bud.compiled_depth:>14}")

  Size   Fidelity depth    Depth depth   Budget depth
-------------------------------------------------------
     4               34             34             41


     6               54             54             65


     8               74             74             89


    10               94             94            113


    12              114            114            137


## Summary

- **5 built-in strategies** cover the full trade-off spectrum: fidelity,
  depth, speed, cost, and budget.
- **`QBCompiler.compile()`** supports three strategies directly:
  `fidelity_optimal`, `depth_optimal`, and `budget_optimal`.
  Pass them via `compiler.compile(circuit, strategy="depth_optimal")`.
- **`speed_optimal` and `cost_optimal`** are available through the
  lower-level API: `get_strategy(name)` → `strategy.build_pass_manager(config)`.
- Use `get_strategy(name)` for convenient lookup, or instantiate directly.
- `BudgetAwareStrategy` is unique: it takes `budget_usd` and `shots`
  parameters and enforces cost constraints.
- Custom strategies are straightforward: subclass `CompilationStrategy`,
  implement `build_pass_manager()`, and compose passes from the available
  building blocks.
- `PassManager.describe()` provides a human-readable summary of any
  strategy's pass pipeline.